# 30秒快剪-JFEX展会速览：Google Colab 自动视频生成器

这个 Notebook 是完整的 Jupyter/Colab JSON 文件，不只是占位文件。按顺序运行所有单元格后，会从 Google Drive 的素材目录中自动扫描视频、照片、logo 和音乐，并输出 30 秒展会宣传片。

默认素材目录：`/content/drive/MyDrive/30秒快剪-JFEX展会速览`

输出文件：`output/final_30s_exhibition_video.mp4`

素材分析表：`output/material_analysis.csv`

运行顺序建议：从菜单选择 `Runtime` → `Run all`。如果某个单元格报错，请先看错误信息中提示的素材目录、依赖安装或 `generate_video.py` 路径问题。


## Cell 1：挂载 Google Drive

第一次运行时，Colab 会弹出授权链接或授权窗口。请登录你的 Google 账号并允许 Colab 访问 Drive。

In [1]:
from google.colab import drive

drive.mount('/content/drive')
print('✅ Google Drive 已挂载到 /content/drive')


Mounted at /content/drive
✅ Google Drive 已挂载到 /content/drive


## Cell 2：安装依赖

安装视频生成需要的基础库：MoviePy、OpenCV、Pillow 和 imageio-ffmpeg。这里直接安装用户要求的包名，确保普通 Colab 环境可运行。

In [2]:
!!python -m pip install -q moviepy==1.0.3 opencv-python pillow imageio-ffmpeg numpy

import importlib
for package_name, import_name in [
    ('moviepy', 'moviepy'),
    ('opencv-python', 'cv2'),
    ('pillow', 'PIL'),
    ('imageio-ffmpeg', 'imageio_ffmpeg'),
    ('numpy', 'numpy'),
]:
    importlib.import_module(import_name)
    print(f'✅ {package_name} 可用')


✅ moviepy 可用
✅ opencv-python 可用
✅ pillow 可用
✅ imageio-ffmpeg 可用
✅ numpy 可用


## Cell 3：设置素材路径

本单元格会设置素材根目录，并自动创建 `video/`、`photo/`、`logo/`、`music/`、`output/` 五个子文件夹。你不需要提前创建不存在的目录。

In [3]:
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/30秒快剪-JFEX展会速览')
VIDEO_DIR = ROOT / 'video'
PHOTO_DIR = ROOT / 'photo'
LOGO_DIR = ROOT / 'logo'
MUSIC_DIR = ROOT / 'music'
OUTPUT_DIR = ROOT / 'output'

for folder in [VIDEO_DIR, PHOTO_DIR, LOGO_DIR, MUSIC_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('✅ 素材根目录：', ROOT)
print('✅ video 目录：', VIDEO_DIR)
print('✅ photo 目录：', PHOTO_DIR)
print('✅ logo 目录：', LOGO_DIR)
print('✅ music 目录：', MUSIC_DIR)
print('✅ output 目录：', OUTPUT_DIR)


✅ 素材根目录： /content/drive/MyDrive/30秒快剪-JFEX展会速览
✅ video 目录： /content/drive/MyDrive/30秒快剪-JFEX展会速览/video
✅ photo 目录： /content/drive/MyDrive/30秒快剪-JFEX展会速览/photo
✅ logo 目录： /content/drive/MyDrive/30秒快剪-JFEX展会速览/logo
✅ music 目录： /content/drive/MyDrive/30秒快剪-JFEX展会速览/music
✅ output 目录： /content/drive/MyDrive/30秒快剪-JFEX展会速览/output


In [4]:
print(ROOT)
print(VIDEO_DIR)
print(PHOTO_DIR)
print(LOGO_DIR)
print(MUSIC_DIR)

/content/drive/MyDrive/30秒快剪-JFEX展会速览
/content/drive/MyDrive/30秒快剪-JFEX展会速览/video
/content/drive/MyDrive/30秒快剪-JFEX展会速览/photo
/content/drive/MyDrive/30秒快剪-JFEX展会速览/logo
/content/drive/MyDrive/30秒快剪-JFEX展会速览/music


## Cell 4：扫描 video/photo/logo/music 文件夹

本单元格会打印当前发现的素材。程序不会假设固定文件名，只根据扩展名识别素材。

In [5]:
from pathlib import Path

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.m4v', '.avi', '.mkv', '.webm'}
AUDIO_EXTENSIONS = {'.mp3', '.wav', '.m4a', '.aac', '.flac', '.ogg'}

def scan_files(folder: Path, extensions):
    extensions = {ext.lower() for ext in extensions}
    if not folder.exists():
        folder.mkdir(parents=True, exist_ok=True)
        return []
    return sorted(path for path in folder.rglob('*') if path.is_file() and path.suffix.lower() in extensions)

video_files = scan_files(VIDEO_DIR, VIDEO_EXTENSIONS)
photo_files = scan_files(PHOTO_DIR, IMAGE_EXTENSIONS)
logo_files = scan_files(LOGO_DIR, IMAGE_EXTENSIONS)
music_files = scan_files(MUSIC_DIR, AUDIO_EXTENSIONS)

for title, files in [('视频', video_files), ('照片', photo_files), ('logo', logo_files), ('音乐', music_files)]:
    print(f'\n{title}：')
    if files:
        for file in files:
            print(' -', file.relative_to(ROOT))
    else:
        print('（未发现，将自动跳过或使用其他素材替代）')



视频：
 - video/人流视频.mp4
 - video/展馆内视频.mp4

照片：
 - photo/人流照片.jpg
 - photo/企业展位.jpg
 - photo/展馆入口.jpg

logo：
 - logo/JFEX-logo.png

音乐：
（未发现，将自动跳过或使用其他素材替代）


In [6]:
!git clone https://github.com/sherylling1986-beep/solid-octo-pancake.git

Cloning into 'solid-octo-pancake'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 29 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 22.52 KiB | 411.00 KiB/s, done.
Resolving deltas: 100% (9/9), done.


In [7]:
import os

print(os.listdir('/content'))

print("\n检查仓库：")
print(os.listdir('/content/solid-octo-pancake'))

['.config', 'drive', 'solid-octo-pancake', 'sample_data']

检查仓库：
['Generate_30s_Exhibition_Video.ipynb', '.git', 'src', 'requirements.txt', '.gitkeep', 'README.md', '.gitignore', 'generate_video.py']


In [8]:
import os
from pathlib import Path

print("当前 /content 内容：", os.listdir("/content"))

print("检查 generate_video.py 是否存在：")
print(Path("/content/solid-octo-pancake/generate_video.py").exists())

print("检查素材根目录是否存在：")
print(ROOT)
print(ROOT.exists())

print("检查素材子目录：")
for p in [VIDEO_DIR, PHOTO_DIR, LOGO_DIR, MUSIC_DIR, OUTPUT_DIR]:
    print(p, p.exists())

当前 /content 内容： ['.config', 'drive', 'solid-octo-pancake', 'sample_data']
检查 generate_video.py 是否存在：
True
检查素材根目录是否存在：
/content/drive/MyDrive/30秒快剪-JFEX展会速览
True
检查素材子目录：
/content/drive/MyDrive/30秒快剪-JFEX展会速览/video True
/content/drive/MyDrive/30秒快剪-JFEX展会速览/photo True
/content/drive/MyDrive/30秒快剪-JFEX展会速览/logo True
/content/drive/MyDrive/30秒快剪-JFEX展会速览/music True
/content/drive/MyDrive/30秒快剪-JFEX展会速览/output True


## Cell 5：输出素材分析结果 material_analysis.csv

本单元格会调用 `generate_video.py` 中的素材分析函数：

- 视频：读取时长、分辨率、清晰度评分，并自动截取关键帧到 `output/keyframes/`
- 照片：读取尺寸，并评估是否适合 16:9 视频展示
- logo：按图片处理，并标记为品牌结束页候选素材

分析结果会写入 `output/material_analysis.csv`。

In [9]:
import sys
from pathlib import Path
import importlib

REPO_DIR = Path("/content/solid-octo-pancake")

sys.path.insert(0, str(REPO_DIR))

print("Python搜索路径已加入：", REPO_DIR)
print("generate_video.py 是否存在：", (REPO_DIR / "generate_video.py").exists())

import generate_video
importlib.reload(generate_video)

print("✅ generate_video 导入成功")

generate_video.ensure_directories(ROOT)
assets = generate_video.analyze_materials(ROOT)
analysis_csv = generate_video.write_csv(ROOT, assets)

print(f'✅ 素材分析完成，共发现 {len(assets)} 个视频/照片/logo 素材')
print('✅ material_analysis.csv：', analysis_csv)
print('\n前 20 条分析结果：')
for asset in assets[:20]:
    print(
        f'{asset.media_type:5s} | {asset.path.name} | '
        f'{asset.width}x{asset.height} | duration={asset.duration}s | '
        f'score={asset.quality} | 用途={asset.purpose}'
    )


Python搜索路径已加入： /content/solid-octo-pancake
generate_video.py 是否存在： True


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



✅ generate_video 导入成功
✅ 素材分析完成，共发现 6 个视频/照片/logo 素材
✅ material_analysis.csv： /content/drive/MyDrive/30秒快剪-JFEX展会速览/output/material_analysis.csv

前 20 条分析结果：
photo | 企业展位.jpg | 4032x3024 | duration=0.0s | score=100.0 | 用途=booth
photo | 展馆入口.jpg | 1920x2560 | duration=0.0s | score=81.41 | 用途=entrance
photo | 人流照片.jpg | 1280x1707 | duration=0.0s | score=75.59 | 用途=crowd
video | 展馆内视频.mp4 | 1080x1920 | duration=52.05s | score=68.36 | 用途=entrance
video | 人流视频.mp4 | 1080x1920 | duration=12.75s | score=54.45 | 用途=crowd
logo  | JFEX-logo.png | 444x133 | duration=0.0s | score=45.83 | 用途=scene/logo


In [6]:
!pip show moviepy

Name: moviepy
Version: 2.2.1
Summary: Video editing with Python
Home-page: 
Author: Zulko 2024
Author-email: 
License: MIT License
Location: /usr/local/lib/python3.12/dist-packages
Requires: decorator, imageio, imageio_ffmpeg, numpy, pillow, proglog, python-dotenv
Required-by: 


In [7]:
!pip uninstall moviepy -y
!pip install moviepy==1.0.3

Found existing installation: moviepy 2.2.1
Uninstalling moviepy-2.2.1:
  Successfully uninstalled moviepy-2.2.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.3/388.3 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for moviepy: filename=moviepy-1.0.3-py3-none-any.whl size=110712 sha256=adbc51cd1dd485ffbaecd373e9e7ceb8ed885afadceaeaba4ddd59258edfc28a
  Stored in directory: /root/.cache/pip/wheels/df/ba/4b/0917fc0c8833c8ba7016565fc975b74c67bc8610806e930272
Successfully built moviepy


## Cell 6：自动生成 30 秒视频

本单元格会按以下结构生成视频：

| 时间 | 内容 | 优先素材 |
| --- | --- | --- |
| 0-3 秒 | 开场 | 展馆入口、展会现场、人流画面 |
| 3-10 秒 | 展位展示 | 企业展位、展馆环境 |
| 10-20 秒 | 产品展示 | 产品图片、产品视频 |
| 20-27 秒 | 商务交流 | 客户交流、人流、活动现场 |
| 27-30 秒 | 品牌结束页 | logo 图片 |

如果某个类别没有素材，会自动使用其他质量评分最高的素材替代。

In [10]:
import generate_video

final_video_path, analysis_csv = generate_video.build_video(ROOT)

print('✅ 30 秒视频已生成')
print('视频路径：', final_video_path)
print('分析表：', analysis_csv)


当前发现的素材：

视频：
video/人流视频.mp4
video/展馆内视频.mp4

照片：
photo/人流照片.jpg
photo/企业展位.jpg
photo/展馆入口.jpg

logo：
logo/JFEX-logo.png

音乐：
（未发现，将自动跳过或使用其他素材替代）

素材分析表已生成：/content/drive/MyDrive/30秒快剪-JFEX展会速览/output/material_analysis.csv
开场: 使用 展馆入口.jpg（entrance, 评分 81.41）
展位展示: 使用 企业展位.jpg（booth, 评分 100.0）
产品展示: 使用 人流照片.jpg（crowd, 评分 75.59）
商务交流: 使用 人流视频.mp4（crowd, 评分 54.45）
品牌结束页: 使用 JFEX-logo.png（scene/logo, 评分 45.83）
Moviepy - Building video /content/drive/MyDrive/30秒快剪-JFEX展会速览/output/final_30s_exhibition_video.mp4.
MoviePy - Writing audio in final_30s_exhibition_videoTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/drive/MyDrive/30秒快剪-JFEX展会速览/output/final_30s_exhibition_video.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/30秒快剪-JFEX展会速览/output/final_30s_exhibition_video.mp4
✅ 30 秒视频已生成
视频路径： /content/drive/MyDrive/30秒快剪-JFEX展会速览/output/final_30s_exhibition_video.mp4
分析表： /content/drive/MyDrive/30秒快剪-JFEX展会速览/output/material_analysis.csv


## Cell 7：输出最终视频路径并在 Colab 中预览

最终视频会保存到 Google Drive 的 `output/` 文件夹中。你可以直接在 Drive 中下载 MP4，也可以在下方播放器中预览。

In [16]:
from IPython.display import HTML, display

final_video = OUTPUT_DIR / 'final_30s_exhibition_video.mp4'
analysis_csv = OUTPUT_DIR / 'material_analysis.csv'

print('✅ 最终视频：', final_video)
print('✅ 素材分析 CSV：', analysis_csv)

if final_video.exists():
    html = f'''<video width="960" controls>
      <source src="{final_video.as_posix()}" type="video/mp4">
      你的浏览器不支持 video 标签，请到 Google Drive output 文件夹下载 MP4。
    </video>'''
    display(HTML(html))
else:
    print('⚠️ 尚未找到最终视频，请先运行 Cell 6。')


✅ 最终视频： /content/drive/MyDrive/30秒快剪-JFEX展会速览/output/final_30s_exhibition_video.mp4
✅ 素材分析 CSV： /content/drive/MyDrive/30秒快剪-JFEX展会速览/output/material_analysis.csv


## 常见问题

### 1. Notebook 找不到 `generate_video.py` 怎么办？

请确认 `Generate_30s_Exhibition_Video.ipynb` 和 `generate_video.py` 在同一个 Colab 工作目录中。如果你是从 GitHub 克隆项目，请先 `cd` 到项目根目录再运行 Notebook，或把 `generate_video.py` 上传到 Colab 的当前文件区。

### 2. 没有 logo 或音乐是否可以生成？

可以。没有 logo 时，最后 3 秒会使用质量最高的照片/视频素材替代；没有音乐时，程序会保留视频原声。

### 3. 素材文件名必须固定吗？

不需要。程序会扫描文件夹内所有支持格式的文件，并根据文件名关键词、分辨率、清晰度评分和视频时长自动选择。

### 4. 为什么 Notebook 文件要超过 10KB？

为了避免只生成占位文件或不完整文件，本 Notebook 包含完整的说明、路径设置、扫描逻辑、分析调用、视频生成调用与预览逻辑，因此文件体积会明显大于 10KB。

### 5. 如果素材很少怎么办？

至少需要 `video/` 或 `photo/` 中有一个可用素材。缺少 logo、music 或某类主题素材时，脚本会自动使用已有的高质量素材补位。
